# 01 – d1-3B gate on dev set (Kaggle)

> **Provenance note:** reconstructed from the exact Kaggle cell code used in the experiment (the original notebook was not saved to disk). Final run used `AISIS_DevSet_v1.xlsx` + `gate_questions_v2.json` (v1 file for the earlier run); the pilot run used the 60-row pilot file. Output → `results/gate_dev/`.


In [ ]:
!pip install -q -U "transformers>=5.14" accelerate

In [ ]:
import glob, json, os, time
import pandas as pd

xlsx = glob.glob("/kaggle/input/**/AISIS_DevSet_v1.xlsx", recursive=True)[0]
qjson = glob.glob("/kaggle/input/**/gate_questions_v2.json", recursive=True)[0]
print(xlsx)

dev = pd.read_excel(xlsx, sheet_name="DevSet").fillna("")
QUESTIONS = json.load(open(qjson, encoding="utf-8"))["questions"]
TOOLS = {k: v for k, v in QUESTIONS["tool"]["criteria"].items() if k != "none"}

instances = []
for r in dev.to_dict("records"):
    base = dict(dev_id=r["dev_id"], type=r["type"], topic=r["topic"],
                gold_action=r["label_action"], gold_supported=r["label_supported"],
                gold_specific=r["label_specific"], gold_emergency=r["label_emergency"],
                gold_tool=r["label_tool"])
    instances.append({**base, "version": "standard", "query": r["query_standard"], "passage": r["passage_text"]})
    instances.append({**base, "version": "dialect", "query": r["query_dialect_synth"], "passage": r["passage_text"]})
    if r["type"] == "answerable":
        instances.append({**base, "version": "wrong_passage", "query": r["query_standard"],
                          "passage": r["wrong_passage_text"],
                          "gold_action": "cannot_answer", "gold_supported": "not_supported"})
print(len(instances), "instances")

In [ ]:
import torch
from transformers import AutoModel

def load(dtype):
    return AutoModel.from_pretrained("LiquidAI/d1-3B", trust_remote_code=True, dtype=dtype).to("cuda")

model = load(torch.float16)
test = model.system_one("test", {"q": {"type": "choice", "instructions": "pick", "criteria": {"a": "first", "b": "second"}}})
p = test["answers"]["q"]["probabilities"]
vals = list(p.values()) if isinstance(p, dict) else list(p)
if any(v != v for v in vals):  # NaN -> fp16 failed
    del model; torch.cuda.empty_cache()
    model = load(torch.float32); print("fell back to fp32")

def predict(state, questions):
    res = model.system_one(state, questions)
    return res["answers"], res
# If OOM on re-run: Restart session, or: del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
def state_of(inst):
    return {"user_query": inst["query"], "retrieved_passage": inst["passage"], "available_tools": TOOLS}

rows, t0 = [], time.time()
for k, inst in enumerate(instances):
    answers, raw = predict(state_of(inst), QUESTIONS)
    if k == 0:
        print(raw)
    row = {x: inst[x] for x in ["dev_id", "type", "topic", "version", "gold_action", "gold_supported",
                                "gold_specific", "gold_emergency", "gold_tool"]}
    for q in QUESTIONS:
        a = answers[q]
        row[f"pred_{q}"] = a.get("choice")
        row[f"conf_{q}"] = a.get("confidence")
        row[f"probs_{q}"] = json.dumps(a.get("probabilities"), ensure_ascii=False, default=str)
    rows.append(row)
elapsed = time.time() - t0
print(f"{elapsed:.1f}s total, {1000*elapsed/len(instances):.0f} ms per instance (5 questions each)")
try:
    import torch
    print("peak GPU memory GB:", torch.cuda.max_memory_allocated() / 1e9)
except Exception:
    pass
res = pd.DataFrame(rows)
res.to_csv("/kaggle/working/results_d1_dev.csv", index=False)

In [ ]:
res["ok_action"] = res.pred_action == res.gold_action
res["ok_supported"] = res.pred_supported == res.gold_supported
res["ok_specific"] = res.pred_specific == res.gold_specific
res["ok_emergency"] = res.pred_emergency == res.gold_emergency
tool_rows = res[res.gold_action == "call_tool"].copy()
tool_rows["ok_tool"] = tool_rows.pred_tool == tool_rows.gold_tool
print(res.groupby("version")[["ok_action", "ok_supported", "ok_specific", "ok_emergency"]].mean().round(3))
print("\nTool accuracy (call_tool rows):\n", tool_rows.groupby("version").ok_tool.mean().round(3))
print("\nAction accuracy by type:\n", res[res.version != "wrong_passage"].groupby(["type", "version"]).ok_action.mean().round(3).unstack())